# Multilevel (Quasi-)Monte Carlo for Asian Options

Compares MLMC, CubMLMCCont, MLQMC, and CubMLQMCCont for pricing an Asian call option under geometric Brownian motion.

In [ ]:
using QMC
import QMC: Uniform
using Statistics

## Single-Level Reference Values

Compute the Asian option price at increasing time-step resolutions using single-level QMC.

In [ ]:
let
    for level in 0:4
        d_steps = 2 * 2^level
        dd_level = DigitalNetB2(d_steps; seed=7)
        tm_level = GeometricBrownianMotion(dd_level;
            volatility=0.2, start_price=100.0,
            interest_rate=0.05, t_final=1.0)
        fo_level = FinancialOption(tm_level; option_type=:asian)
        sc_level = CubQMCNetG(fo_level; abs_tol=1e-4)
        result_level = integrate(sc_level)
        println("$(lpad(d_steps, 3)) time steps: $(round(result_level.solution, digits=5))")
    end
end

## Multilevel Comparison

Compare four multilevel algorithms at decreasing tolerances.

In [ ]:
# Setup MC and QMC versions
function compare_ml(abs_tol)
    dd_mc  = IIDStdUniform(16; seed=7)
    dd_qmc = DigitalNetB2(16; seed=7, replications=8)
    tm_mc  = GeometricBrownianMotion(dd_mc;  volatility=0.2, start_price=100.0,
                interest_rate=0.05, t_final=1.0)
    tm_qmc = GeometricBrownianMotion(dd_qmc; volatility=0.2, start_price=100.0,
                interest_rate=0.05, t_final=1.0)
    fml_mc  = FinancialOptionML(tm_mc;  d_coarsest=4)
    fml_qmc = FinancialOptionML(tm_qmc; d_coarsest=4)

    results = Dict{String, Any}()
    for (name, sc) in [
        ("MLMC",      CubMLMC(fml_mc;      abs_tol=abs_tol)),
        ("MLMCCont",  CubMLMCCont(fml_mc;  abs_tol=abs_tol)),
        ("MLQMC",     CubMLQMC(fml_qmc;    abs_tol=abs_tol)),
        ("MLQMCCont", CubMLQMCCont(fml_qmc; abs_tol=abs_tol)),
    ]
        r = integrate(sc)
        n_used = get(r.data, :n_total, get(r.data, :n, 0))
        results[name] = (solution=r.solution, n=n_used)
        println("  $name: sol=$(round(r.solution, digits=5)), n=$n_used")
    end
    return results
end

for tol in [1e-1, 5e-2]
    println("\nabs_tol = $tol")
    compare_ml(tol)
end